# 04. Dựng đồ thị chính thức (task 1.5, 1.6)

- **NF-ToN-IoT-v3** (bộ chính, D1): W = S = 5 s, chia `blocked` 10 khối, gap = 1 cửa sổ (D2, D3).
- **NF-ToN-IoT-v2** (bộ phụ): W = S = 1000 flow, cùng cách chia (ts = số thứ tự dòng, giả định theo thời gian).

Mỗi bộ ghi `graphs_*.pt`, `split.json`, `scaler.pkl`, `summary.json` vào `/kaggle/working/processed/<bộ>/`. Output của kernel này dùng làm đầu vào cho các notebook sau qua `kernel_sources`. Chạy trong tiến trình con để `pip` không làm hỏng import của kernel.

In [ ]:
import glob, os, subprocess, sys, time

def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-5000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode != 0:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")

REPO = "/tmp/gnn-iot-ids"          # ngoài /kaggle/working để không lẫn vào output
run(["rm", "-rf", REPO])
run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
run(["git", "-C", REPO, "log", "-1", "--format=%H %s"])
with open(f"{REPO}/requirements.txt") as f:
    reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
run([sys.executable, "-m", "pip", "install", "-q", *keep])

In [ ]:
OUT = "/kaggle/working/processed"
V3 = next(glob.glob("/kaggle/input/**/NF-ToN-IoT-v3.csv", recursive=True).__iter__())
t0 = time.time()
run([sys.executable, "-I", "scripts/build_dataset_cli.py", "nf_ton_iot_v3", V3, "5", "5", "blocked", "1",
     f"{OUT}/nf_ton_iot_v3", "nf_ton_iot_v3_w5s5_blocked"], cwd=REPO)
print(f"v3 xong sau {time.time() - t0:.0f} s")

In [ ]:
V2 = next(glob.glob("/kaggle/input/**/NF-ToN-IoT-v2.csv", recursive=True).__iter__())
t0 = time.time()
run([sys.executable, "-I", "scripts/build_dataset_cli.py", "nf_ton_iot_v2", V2, "1000", "1000", "blocked", "1",
     f"{OUT}/nf_ton_iot_v2", "nf_ton_iot_v2_w1000s1000_blocked"], cwd=REPO)
print(f"v2 xong sau {time.time() - t0:.0f} s")
run(["du", "-sh", OUT])